# ECHOSCAPE — Data Evaluation**What this notebook is for:** looking hard at the Sentinel-1 data *before* anylabelling or training happens.Two reasons this is worth a day:1. Some of these checks can tell us the project will not work as planned — for   example, that radar cannot separate six classes. Far better to find that out   now than after a week of hand-labelling.2. The brief requires notebooks with executed results. This is one of them.**Nothing here needs labels and nothing here needs a trained model.** It is alldone on the radar alone.---### Before running anything- Data must be at **50 m** (`TARGET_RES_M = 50` in `config.py`).- All three areas downloaded, all twelve months: `mannheim`, `karlsruhe`,  `odenwald`.- Files live in `data/s1_monthly/<area>/<area>_2021-<mm>_50m.tif`.- Each file has 3 bands: **1 = VV**, **2 = VH**, **3 = number of passes**.- Values are linear gamma0. Convert to decibels with `10 * log10(value)` before  looking at anything or plotting.### PriorityIf time is short, do **Section 5**, **Section 7** and **Section 3**. Those threechange decisions. The rest is supporting evidence.

In [ ]:
# SETUP# - add the project root to sys.path so `import config` works from this folder# - import: numpy, rasterio, matplotlib, pathlib# - sklearn needed later for Section 5 (KMeans, silhouette_score) - not installed#   yet, add it to requirements.txt# - write a small helper: load(area, month) -> vv, vh, passes# - write a small helper: to_db(x) -> 10 * log10(x), with zeros masked out

---# Group 1 — Is the data any good?Basic quality checks. Cheap to run, and they catch the problems that otherwisesurface much later disguised as modelling failures.

## 1. Coverage check**Question:** did the satellite actually see every pixel, every month?**What to do:** read band 3 of every file. For each area and month, report thepercentage of pixels with at least one pass, and the typical number of passes.**Why it matters:** any pixel with zero passes must be excluded from the areapercentages later. If it is not, months with gaps quietly deflate every classand invent month-to-month change that is not real. This is the single mostlikely wrong number in the whole submission.**What good looks like:** 100% coverage, 5–8 passes everywhere.**If it is bad:** note which months are affected and carry that through to theprototype.

In [ ]:
# 1. COVERAGE# - loop over the 3 areas x 12 months# - band 3 = passes per pixel# - table: area | month | % pixels with >=1 pass | median passes | min passes# - flag any month below 100% coverage# - small heatmap: area (rows) x month (columns), coloured by coverage

## 2. Brightness over time**Question:** does the signal move smoothly through the year, or does it jump?**What to do:** for each area, plot the median VV and VH in decibels across thetwelve months.**Why it matters:** a smooth seasonal curve means the data is consistent and thechanges are real. A sudden step means something changed in the processing, noton the ground — and the model would learn that step as if it were land coverchanging.**What good looks like:** gentle seasonal movement, a couple of decibels.Farmland areas should show more movement than forest.**If it is bad:** a jump of 3+ dB between neighbouring months with no physicalexplanation needs investigating before training.

In [ ]:
# 2. BRIGHTNESS OVER TIME# - for each area: median VV dB and median VH dB per month# - line plot, 12 points, one line per area, VV and VH as two panels# - also plot the VV/VH ratio - it is less sensitive to moisture than either alone# - look for: smooth curves, no steps, forest flatter than farmland

## 3. Noise floor  **(priority)****Question:** how much does the picture change when nothing on the ground haschanged?**What to do:** take two individual passes from the same month, a few daysapart. Subtract one from the other. Measure the spread of the difference.**Why it matters:** the brief asks for month-to-month change in percentagepoints. Any change smaller than this number is noise, not land cover. Withoutit, the headline output of the prototype is unfalsifiable — we would have no wayof saying which changes mean anything.This number goes directly into the report and into the prototype's interface.**What good looks like:** a difference centred on zero. The width of thatdistribution *is* the noise floor.**Note:** this needs two single passes, not the monthly composite. `fetch_s1.py`currently only saves the composite, so it will need a small addition — fetch twoscenes separately for one month and keep them.

In [ ]:
# 3. NOISE FLOOR# - pick one area, one month with >= 6 passes# - fetch two individual passes separately (needs a small change to fetch_s1.py,#   or do it inline here with the same data-API call)# - difference in dB: pass_A_db - pass_B_db# - histogram of the difference; report mean (should be ~0) and std# - the std is the noise floor - WRITE THIS NUMBER DOWN, it is needed in Phase 6# - repeat for the monthly composites of two neighbouring months for comparison

## 4. Outliers and clipping**Question:** how extreme do the bright pixels get, and where do we cut them off?**What to do:** find the 1st and 99th percentile of VV and VH across all areas,and the absolute maximum.**Why it matters:** metal structures act like mirrors and return enormous values.One pixel measured 757 when the typical value is 0.12. Left alone, those fewpixels dominate any scaling we apply and waste most of the numeric range.**What to decide:** the clipping limits to use when preparing training data.Roughly VV in [-25, 5] dB and VH in [-30, 0] dB, but confirm from the numbers.

In [ ]:
# 4. OUTLIERS# - pooled histogram of VV dB and VH dB across all areas and months# - report p1, p50, p99, max for each# - count how many pixels sit above p99.9# - decide clipping limits and write them into config.py as VV_CLIP, VH_CLIP

---# Group 2 — Is there enough information in the data?This is the group that can change the plan. These checks ask whether radar cando the job at all, and they do it **without labels**, so they can be run beforeany labelling effort is spent.

## 5. Clustering without labels  **(priority — do this first)****Question:** how many land-cover classes does the data naturally support?**What to do:** take a sample of pixels as [VV dB, VH dB, VV/VH]. Run k-meansfor k = 2, 3, 4, 5, 6, 7, 8. For each k, measure how well separated the clustersare (silhouette score). Plot score against k.Then map the clusters back onto the image and look at them.**Why it matters most:** the plan commits to six classes. If the data onlysupports three or four well-separated groups, six is wishful thinking and wewould discover it after days of labelling. This finds out in an hour.**What good looks like:** a clear peak in the score. If it peaks at 4, that istelling us something important.**Then look at the map.** Do the clusters correspond to recognisable things —rivers, the city, the forest block? If the clusters look like real land cover,the signal is there. If they look like noise, it is not.**What to do with the answer:** if the data supports fewer classes than planned,merge the hard pairs — cropland with grass is the expected casualty — and say soin the report, with this chart as the justification. That is a much strongerreport than reporting a poor score for six classes and not explaining why.

In [ ]:
# 5. CLUSTERING WITHOUT LABELS# - sample ~100k random pixels from one summer month, all three areas# - features: VV dB, VH dB, VV/VH dB   (standardise them first)# - KMeans for k in 2..8, record silhouette_score for each# - plot silhouette vs k - look for the peak# - for the best k AND for k=6 (our planned number):#     - map the cluster labels back to image shape and display#     - do the clusters look like water / city / forest / fields?# - DECISION TO RECORD: how many classes is the data actually able to support?

## 6. Histogram shape**Question:** can we see separate groups in the raw numbers?**What to do:** plot the distribution of VV in decibels for a whole area. Thenplot VV against VH as a 2D density.**Why it matters:** a simpler version of Section 5 and a good sanity check on it.Water should appear as a distinct dark bump. Buildings as a bright tail.**What good looks like:** at least two or three visible bumps. A single smoothhill means the classes overlap almost completely.

In [ ]:
# 6. HISTOGRAM SHAPE# - histogram of VV dB for one area, one month - look for bumps# - 2D histogram / hexbin of VV dB vs VH dB - look for separate blobs# - overlay the three areas to compare (forest area should differ from urban)

## 7. Does the seasonal pattern separate crops?  **(priority)****Question:** does cropland reveal itself over time, even though it is invisibleon any single date?**What to do:** take each pixel's twelve monthly VV values as a short curve.Sample a few thousand pixels. Cluster the *curves*. Plot the average curve ofeach cluster.**Why it matters:** we measured that cropland and grass overlap 71% at 10 m and61% at 50 m on a single date — they are genuinely hard to tell apart. The wholeargument for a three-month input in Phase 5 is that crops have a distinctiveshape over the year: bare soil, then growing, then harvested, then bare again.Forest stays flat.This tests that argument directly, before building the model that depends on it.**What good looks like:** one cluster with a pronounced hump in the growingseason and dips either side. That is cropland showing itself.**If flat curves only:** the three-month experiment will not help either, andcropland and grass should be merged. Finding that out now saves an entiretraining run and a wrong conclusion in the report.

In [ ]:
# 7. SEASONAL CURVES# - build a stack: 12 months x H x W for one area (VV dB)# - sample ~5000 pixels that are valid in all 12 months# - each sample is a 12-value curve; standardise each curve (subtract its own mean)#   so we cluster on SHAPE, not on brightness# - KMeans on the curves, k = 4 or 5# - plot the mean curve of each cluster, with month on the x axis# - map clusters back to the image and look# - DECISION TO RECORD: is there a crop-shaped curve? does the 3-month experiment#   in Phase 5 have a basis?

---# Group 3 — What does this tell us about how to build the model?These produce numbers that would otherwise be chosen by guesswork.

## 8. How far apart are two pixels before they are independent?**Question:** over what distance does the image stop repeating itself?**What to do:** measure how similarity between pixels falls off as the distancebetween them grows. Find the distance at which it drops to near zero.**Why it matters, twice over:**1. **It sets the tile size.** Tiles should be comfortably larger than this   distance, or each tile contains barely more information than a single point.2. **It proves the train/test split is honest.** If pixels stay similar over   2 km, then two tiles cut 500 m apart are nearly the same tile. Splitting   randomly by tile would leak the test set into training and inflate the score.   The plan already splits by whole area — this gives the number that justifies   it, rather than asserting it.**What good looks like:** a curve that decays over a few hundred metres to acouple of kilometres. Whatever it is, record it.

In [ ]:
# 8. SPATIAL CORRELATION LENGTH# - take one area, one month, VV dB# - compute correlation between the image and itself shifted by d pixels,#   for d = 1, 2, 4, 8, 16, 32, 64 (both directions, average them)# - plot correlation against distance in metres (d * 50)# - find the distance where correlation drops below ~0.2# - NUMBER TO RECORD: correlation length in metres# - check: is our 128 px tile (6.4 km) comfortably bigger than it? (should be)

## 9. Are the three areas comparable?**Question:** is holding out one area a fair test, or a different world?**What to do:** compare the distributions of VV, VH and the ratio between`mannheim`, `karlsruhe` and `odenwald`.**Why it matters:** one area is held back entirely for testing. If it is nearlyidentical to the training areas, the test is too easy and the score isflattering. If it is wildly different, a poor score means a data mismatch ratherthan a bad model, and we would misread it.`odenwald` is mostly forest by design, so it *should* differ — the question isby how much.**What to decide:** which area to hold out. The honest choice is one that isdifferent enough to be a real test but not so different as to be unfair.

In [ ]:
# 9. AREA COMPARISON# - overlay VV dB histograms for the three areas, one summer month# - table: area | median VV | median VH | median ratio | spread# - optional: how much do the distributions overlap, pairwise?# - DECISION TO RECORD: which area is held out for testing, and why

## 10. How much does the viewing angle matter?**Question:** how different does identical ground look from a different orbit?**What to do:** for one area and one month, build two composites — one fromascending passes only, one from descending only. Compare them.**Why it matters:** the pipeline fixes a single orbit track, on the grounds thatmixing viewing geometries makes the same field read differently. This measureshow big that effect actually is, turning a design assumption into a number.**What good looks like:** a visible difference, especially over buildings andsloped ground. The bigger it is, the more the decision is justified.**Bonus:** if the difference is large, that is also a limitation worth stating —the model is learning from one viewing geometry and may not transfer to another.

In [ ]:
# 10. ORBIT GEOMETRY# - same area, same month, build ascending-only and descending-only composites#   (fetch_s1.py currently picks one track; this needs a one-off variation)# - difference image in dB, and its histogram# - report median absolute difference, and where it is worst (city? hills?)# - NUMBER TO RECORD: typical dB difference from geometry alone

---# FindingsFill this in as the sections are completed. This block is what gets copied into`changes.md` and then into the written report, so write it as conclusions, notas notes.| # | Check | Result | What we decided because of it ||---|---|---|---|| 1 | Coverage | | || 2 | Brightness over time | | || 3 | **Noise floor** | | || 4 | Outliers / clipping | | || 5 | **Clusters the data supports** | | || 6 | Histogram shape | | || 7 | **Seasonal curves** | | || 8 | **Correlation length** | | || 9 | Area comparison | | || 10 | Orbit geometry | | |### Decisions that depend on this notebook- **How many classes to label.** Section 5. Do not start labelling before this  is answered — relabelling is expensive, and the class list is the one choice  that is hard to undo.- **Whether the three-month experiment is worth running.** Section 7.- **Tile size, and proof the split is honest.** Section 8.- **Which area is held out.** Section 9.- **Clipping limits for preprocessing.** Section 4.- **The noise floor the prototype quotes.** Section 3.### Housekeeping- `scikit-learn` is needed for Sections 5 and 7 and is not installed yet. Add it  to `requirements.txt`.- Sections 3 and 10 need individual passes rather than monthly composites, so  `fetch_s1.py` needs a small option to keep single scenes.- Run with everything at 50 m. Re-running at a different resolution changes  every number here.